## **Data Engineering Task Notebook**

This notebook is designed to test you through various Data Engineering tasks using the Online Retail II dataset. The tasks will help you develop essential skills in data cleaning, feature engineering, and transforming raw data into valuable insights. You'll explore tasks like handling missing values, aggregating data, creating new features, and performing data joins to simulate real-world data workflows. These tasks will prepare you for more advanced data manipulation and analysis, enhancing your ability to work with large, complex datasets.

# **About the Dataset**
This Online Retail II dataset contains transactional data from a UK-based online retailer selling unique gift-ware. The data covers transactions that occurred between December 1, 2009 and December 9, 2011. The retailer primarily serves both individual customers and wholesalers. The products sold by the company are all-occasion gift items, including home décor, kitchenware, and other unique items.

The dataset includes detailed information on each transaction, providing valuable insights into customer behavior, sales trends, and product performance over time.



## **What can be done with this dataset?**

**Customer Behavior Analysis:** Explore purchasing patterns, repeat customers, and sales volume across different customer segments.

**Sales Forecasting:** Predict future sales by analyzing past transactions, including seasonal trends and demand fluctuations.

**Market Segmentation:** Identify customer groups based on purchase history and demographic data (e.g., by Country).

**Product Performance:** Analyze which products are bestsellers and which have low turnover, and how prices influence sales.

**Time Series Analysis:** Study trends over time, including hourly, daily, and monthly sales volumes, and identify peak shopping periods.

**Anomaly Detection:** Detect potential fraudulent transactions, cancellations, or unusually high sales activity.
Association Rule Mining: Discover products that are often purchased together and identify cross-sell opportunities.



## **Key Attributes in the Dataset:**

**InvoiceNo:** Unique transaction identifier (with cancellations indicated by 'C' prefix).

**StockCode:** Unique product code for each item sold.

**Description:** Name of the product/item sold.

**Quantity:** Quantity of each product sold in the transaction
.
**InvoiceDate:** Date and time of the transaction.

**UnitPrice:** Price per unit of the product.

**CustomerID:** Unique identifier for each customer.

**Country:** The country where the customer resides.

This dataset is a great resource for learning and practicing various data analysis, machine learning, and business intelligence techniques.

## **Exercise**
Complete the following tasks:

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

1. Load the [dataset](https://www.kaggle.com/datasets/lakshmi25npathi/online-retail-dataset) from Kaggle.

In [30]:
path = kagglehub.dataset_download("lakshmi25npathi/online-retail-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\nikol\.cache\kagglehub\datasets\lakshmi25npathi\online-retail-dataset\versions\1


In [31]:
dataset_path = "C:\\Users\\nikol\\.cache\\kagglehub\\datasets\\lakshmi25npathi\\online-retail-dataset\\versions\\1"
files = os.listdir(dataset_path)
print(files)


['online_retail_II.xlsx']


In [32]:
file_path = "C:\\Users\\nikol\\.cache\\kagglehub\\datasets\\lakshmi25npathi\\online-retail-dataset\\versions\\1\\online_retail_II.xlsx"
df = pd.read_excel(file_path)

2. Visualize the dataset and it's structure using appropriate libraries and plots.

In [59]:
print(df.head())

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  Revenue  weekday  \
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom     83.4  Tuesday   
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom     81.0  Tuesday   
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom     81.0  Tuesday   
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom    100.8  Tuesday   
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom     30.0  Tuesday   

   Has Gift  Has Discount Transaction Size  Is Christmas  
0     False         False            Small   

In [34]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 38.8+ MB
None


In [35]:
print(df.isnull().sum())

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64


3. Do some basic cleaning to handle missing values

In [36]:
df['Description'] = df.groupby('StockCode')['Description'].transform(lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x)

#Drop rows where 'Customer ID' is missing
df = df.dropna(subset=['Customer ID'])

print(df.isnull().sum())


Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


In [37]:
print(df.duplicated().sum())

6771


In [38]:
df.drop_duplicates(inplace=True)
print(df.duplicated().sum())

0


4. Create the following features:

Revenue

In [39]:
df["Revenue"] = df["Quantity"] * df["Price"]
print(df[["Quantity", "Price", "Revenue"]].head())


   Quantity  Price  Revenue
0        12   6.95     83.4
1        12   6.75     81.0
2        12   6.75     81.0
3        48   2.10    100.8
4        24   1.25     30.0


DayOfWeek

In [40]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["weekday"] = df["InvoiceDate"].dt.day_name()
print(df["weekday"])


0          Tuesday
1          Tuesday
2          Tuesday
3          Tuesday
4          Tuesday
            ...   
525456    Thursday
525457    Thursday
525458    Thursday
525459    Thursday
525460    Thursday
Name: weekday, Length: 410763, dtype: str


Total Revenue per Customer

In [41]:
customer_revenue = df.groupby("Customer ID")["Revenue"].sum().reset_index()
customer_revenue.columns = ["Customer ID", "Total Revenue"]
print(customer_revenue.head())

   Customer ID  Total Revenue
0      12346.0         -51.74
1      12347.0        1323.32
2      12348.0         222.16
3      12349.0        2646.99
4      12351.0         300.93


Most popular product based on revenue

In [42]:
product_revenue = df.groupby("Description")["Revenue"].sum().reset_index()
product_revenue = product_revenue.sort_values(by="Revenue", ascending=False)
print(product_revenue.head())


                             Description    Revenue
4273  WHITE HANGING HEART T-LIGHT HOLDER  148591.51
3260            REGENCY CAKESTAND 3 TIER  136700.55
259        ASSORTED COLOUR BIRD ORNAMENT   69652.16
2073             JUMBO BAG RED RETROSPOT   51493.35
3056                             POSTAGE   45520.86


Ordersize by summing quantity for each invoiceNo


In [53]:
order_size = df.groupby("Description")["Quantity"].sum().reset_index()
order_size = order_size.sort_values(by="Quantity", ascending=False)
order_size.columns = ["Description", "Total Quantity"]
print(order_size.head())

                             Description  Total Quantity
4273  WHITE HANGING HEART T-LIGHT HOLDER           55760
4380   WORLD WAR 2 GLIDERS ASSTD DESIGNS           54130
692                  BROCADE RING PURSE            47430
2638    PACK OF 72 RETRO SPOT CAKE CASES           44480
259        ASSORTED COLOUR BIRD ORNAMENT           44000


5. Apply a lamda funtions to: 
Segment customers into tiers based on TotalRevenue (e.g., "High", "Medium", "Low").

In [44]:
customer_revenue["Segment"] = customer_revenue["Total Revenue"].apply(
    lambda x: "High" if x >= 1000 else "Medium" if x >= 500 else "Low"
)
print(customer_revenue.head())


   Customer ID  Total Revenue Segment
0      12346.0         -51.74     Low
1      12347.0        1323.32    High
2      12348.0         222.16     Low
3      12349.0        2646.99    High
4      12351.0         300.93     Low


Extract key information from Description and add them as columns (e.g., presence of specific keywords like "Gift" or "Discount"). At least one extra column should be added


In [45]:
df["Has Gift"] = df["Description"].apply(lambda x: "Gift" in str(x))
df["Has Discount"] = df["Description"].apply(lambda x: "Discount" in str(x))

print(df[["Description", "Has Gift", "Has Discount"]].head())


                           Description  Has Gift  Has Discount
0  15CM CHRISTMAS GLASS BALL 20 LIGHTS     False         False
1                   PINK CHERRY LIGHTS     False         False
2                  WHITE CHERRY LIGHTS     False         False
3         RECORD FRAME 7" SINGLE SIZE      False         False
4       STRAWBERRY CERAMIC TRINKET BOX     False         False


Categorize transactions as "Small", "Medium", or "Large" based on Revenue.


In [46]:
df["Transaction Size"] = df["Revenue"].apply(
    lambda x: "Large" if x >= 500 else "Medium" if x >= 100 else "Small"
)
print(df[["Revenue", "Transaction Size"]].head())

   Revenue Transaction Size
0     83.4            Small
1     81.0            Small
2     81.0            Small
3    100.8           Medium
4     30.0            Small



Detect Seasonal Items: Flag items as "Christmas"-themed if the description contains relevant words.


In [47]:
df["Is Christmas"] = df["Description"].apply(lambda x: "Christmas" in str(x))
print(df[["Description", "Is Christmas"]].head())

                           Description  Is Christmas
0  15CM CHRISTMAS GLASS BALL 20 LIGHTS         False
1                   PINK CHERRY LIGHTS         False
2                  WHITE CHERRY LIGHTS         False
3         RECORD FRAME 7" SINGLE SIZE          False
4       STRAWBERRY CERAMIC TRINKET BOX         False


In [48]:
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,weekday,Has Gift,Has Discount,Transaction Size,Is Christmas
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.40,Tuesday,False,False,Small,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.00,Tuesday,False,False,Small,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.00,Tuesday,False,False,Small,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.80,Tuesday,False,False,Medium,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.00,Tuesday,False,False,Small,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525456,538171,22271,FELTCRAFT DOLL ROSIE,2,2010-12-09 20:01:00,2.95,17530.0,United Kingdom,5.90,Thursday,False,False,Small,False
525457,538171,22750,FELTCRAFT PRINCESS LOLA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom,3.75,Thursday,False,False,Small,False
525458,538171,22751,FELTCRAFT PRINCESS OLIVIA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom,3.75,Thursday,False,False,Small,False
525459,538171,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-09 20:01:00,3.75,17530.0,United Kingdom,7.50,Thursday,False,False,Small,False


Classify customers as "Loyal", "Occasional", or "One-time" based on the number of purchases.


In [49]:
# First, we need to calculate the number of purchases for each customer
customer_purchases = df.groupby("Customer ID")["Invoice"].nunique().reset_index()
customer_purchases.columns = ["Customer ID", "Purchase Count"]
# Now, we can classify customers based on their purchase count
customer_purchases["Customer Type"] = customer_purchases["Purchase Count"].apply(
    lambda x: "Loyal" if x >= 5 else "Occasional" if x >= 2 else "One-time"
)
print(customer_purchases.head())

   Customer ID  Purchase Count Customer Type
0      12346.0              15         Loyal
1      12347.0               2    Occasional
2      12348.0               1      One-time
3      12349.0               4    Occasional
4      12351.0               1      One-time


6. Identify multi-item invoices: flag invoices with multiple unique items as "Multi-item Order".


In [50]:
# Check the actual column name (it might be "Invoice" instead of "InvoiceNo")
print("Available columns:", df.columns.tolist())
print("\nLooking for invoice column...")

# Use the correct column name - likely "Invoice" based on the dataset pattern
invoice_items = df.groupby("Invoice")["Description"].nunique().reset_index()
invoice_items.columns = ["Invoice", "Unique Item Count"]
invoice_items["Multi-item Order"] = invoice_items["Unique Item Count"] > 1
print(invoice_items.head())

Available columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue', 'weekday', 'Has Gift', 'Has Discount', 'Transaction Size', 'Is Christmas']

Looking for invoice column...
  Invoice  Unique Item Count  Multi-item Order
0  489434                  8              True
1  489435                  4              True
2  489436                 19              True
3  489437                 23              True
4  489438                 17              True


7. Wrap all fo the above into an ETL pipeline.

In [61]:
def extract(file_path):
    df = pd.read_excel(file_path)
    return df

def transform(df):
    # Clean
    df = df.dropna(subset=['Customer ID'])
    df.drop_duplicates(inplace=True)

    # Features
    df["Revenue"] = df["Quantity"] * df["Price"]
    df["Weekday"] = pd.to_datetime(df["InvoiceDate"]).dt.day_name()

    # Lambda functions
    df["Transaction Size"] = df["Revenue"].apply(
        lambda x: "Large" if x >= 500 else "Medium" if x >= 100 else "Small")

    df["Is Christmas"] = df["Description"].apply(
        lambda x: "Christmas" in str(x))

    df["Has Gift"] = df["Description"].apply(
        lambda x: "Gift" in str(x))

    return df

def load(df):
    print("Pipeline done! Shape:", df.shape)
    return df

def ETL(file_path):
    df = extract(file_path)
    df = transform(df)
    df = load(df)
    return df

# Run the pipeline
df_final = ETL(file_path)
df_final.head()

Pipeline done! Shape: (410763, 13)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Weekday,Transaction Size,Is Christmas,Has Gift
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,Tuesday,Small,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,Tuesday,Small,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,Tuesday,Small,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,Tuesday,Medium,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,Tuesday,Small,False,False
